# 04 -- Statcast Expected Stats Ingestion

Fetch Statcast pitcher and batter expected statistics from Baseball Savant for seasons 2015-2024.

**Requirement:** DATA-04

**Source:** Baseball Savant via `pybaseball.statcast_pitcher_expected_stats()` and `statcast_batter_expected_stats()`

**Key columns:** xwOBA, xBA, xSLG, xISO (pitcher); xwOBA, xBA, xSLG (batter)

**Note:** Statcast data only available from 2015 onward (hardware deployment year). Minimum 50 PA threshold.

In [ ]:
# If editable install is not set up, uncomment the next line:
# import sys; sys.path.insert(0, "..")

from src.data.statcast import fetch_statcast_pitcher, fetch_statcast_batter
from src.data.cache import load_manifest

In [ ]:
# Fetch Statcast pitcher expected stats for all 10 seasons
seasons = range(2015, 2025)
pitcher_dfs = {}
for season in seasons:
    print(f"Fetching pitcher stats {season}...")
    pitcher_dfs[season] = fetch_statcast_pitcher(season)
    print(f"  {len(pitcher_dfs[season])} pitchers")

In [ ]:
# Fetch Statcast batter expected stats for all 10 seasons
batter_dfs = {}
for season in seasons:
    print(f"Fetching batter stats {season}...")
    batter_dfs[season] = fetch_statcast_batter(season)
    print(f"  {len(batter_dfs[season])} batters")

In [ ]:
# Summary: show key Statcast pitcher metrics from 2023
df_p = pitcher_dfs[2023]
print(f"Pitcher columns ({len(df_p.columns)} total): {list(df_p.columns)[:15]}...")
print(f"\nTop 10 pitchers by PA faced (2023):")

# Find the xwOBA column name (may vary: est_woba, xwoba, xwOBA)
xwoba_col = None
for col_name in ["est_woba", "xwoba", "xwOBA"]:
    if col_name in df_p.columns:
        xwoba_col = col_name
        break

if xwoba_col:
    pa_col = "pa" if "pa" in df_p.columns else "PA"
    top_p = df_p.nlargest(10, pa_col) if pa_col in df_p.columns else df_p.head(10)
    display_cols = [c for c in ["last_name", "first_name", pa_col, xwoba_col, "est_ba", "est_slg"] if c in df_p.columns]
    display(top_p[display_cols].reset_index(drop=True))
    print(f"\nxwOBA range: {df_p[xwoba_col].min():.3f} - {df_p[xwoba_col].max():.3f}")
else:
    print("Could not find xwOBA column. Available columns:")
    print(list(df_p.columns))

In [ ]:
# Summary: show key Statcast batter metrics from 2023
df_b = batter_dfs[2023]
print(f"Batter columns ({len(df_b.columns)} total): {list(df_b.columns)[:15]}...")

xwoba_col_b = None
for col_name in ["est_woba", "xwoba", "xwOBA"]:
    if col_name in df_b.columns:
        xwoba_col_b = col_name
        break

if xwoba_col_b:
    pa_col = "pa" if "pa" in df_b.columns else "PA"
    top_b = df_b.nlargest(10, pa_col) if pa_col in df_b.columns else df_b.head(10)
    display_cols = [c for c in ["last_name", "first_name", pa_col, xwoba_col_b, "est_ba", "est_slg"] if c in df_b.columns]
    display(top_b[display_cols].reset_index(drop=True))
    print(f"\nxwOBA range: {df_b[xwoba_col_b].min():.3f} - {df_b[xwoba_col_b].max():.3f}")
else:
    print("Could not find xwOBA column. Available columns:")
    print(list(df_b.columns))

In [ ]:
# Coverage validation: all Statcast-era seasons (2015-2024)
print("Pitcher coverage:")
for season in seasons:
    n = len(pitcher_dfs[season])
    flag = "OK" if n >= 50 else "CHECK"
    print(f"  {season}: {n} pitchers [{flag}]")

print("\nBatter coverage:")
for season in seasons:
    n = len(batter_dfs[season])
    flag = "OK" if n >= 50 else "CHECK"
    print(f"  {season}: {n} batters [{flag}]")

In [ ]:
# Cache check
manifest = load_manifest()
statcast_keys = {k: v for k, v in manifest.items() if k.startswith("statcast_")}
print(f"Cached Statcast files: {len(statcast_keys)}")
for k, v in sorted(statcast_keys.items()):
    print(f"  {k}: {v['row_count']} rows, fetched {v['fetch_date'][:10]}")